# CanastaCL — 03 · Modelado: forecasting de productos volátiles

**Objetivo:** pronosticar el precio semanal de productos con alta volatilidad usando tres enfoques:

1. **Naive** — baseline que predice "el siguiente valor = último observado".
2. **ARIMA** — modelo estadístico clásico con tendencia y autocorrelación.
3. **Prophet** — modelo aditivo de tendencia con cambios automáticos de pendiente.

**Evaluación:** holdout temporal con las últimas 4 semanas como test, métricas MAE / RMSE / MAPE.

## ⚠️ Limitación de honestidad metodológica

El dataset cubre **17 semanas (dic 2025 → abr 2026)**. Esto es **insuficiente** para forecasting serio:

- Prophet idealmente requiere ≥1 año para detectar estacionalidad anual.
- ARIMA con ~13 puntos de entrenamiento da intervalos de confianza muy amplios.
- No es posible validar estacionalidad anual.

**Este notebook es una demo del *flujo* de modelado y evaluación**, no de predicciones de calidad productiva. Las conclusiones se interpretan en ese contexto.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.model import (
    select_series,
    train_test_split_temporal,
    forecast_naive,
    forecast_arima,
    forecast_prophet,
    evaluate,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Carga del dataset limpio

In [ ]:
PROCESSED = PROJECT_ROOT / "data" / "processed" / "precios_clean.parquet"
df = pd.read_parquet(PROCESSED)
print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")
print(f"Rango temporal: {df['fecha_inicio'].min().date()} -> {df['fecha_inicio'].max().date()}")
print(f"Semanas distintas: {df['fecha_inicio'].nunique()}")

## 3. Identificación de series candidatas

Filtros: al menos 15 semanas observadas y alto coeficiente de variación (CV).

In [ ]:
MIN_SEMANAS = 15

agg = (
    df.groupby(["producto_id", "region", "fecha_inicio"], observed=True)["precio_prom"]
      .mean().reset_index()
)
stats = (
    agg.groupby(["producto_id", "region"], observed=True)["precio_prom"]
       .agg(["mean", "std", "count"]).reset_index()
)
stats["cv"] = stats["std"] / stats["mean"]
candidatos = (
    stats[stats["count"] >= MIN_SEMANAS]
    .sort_values("cv", ascending=False)
    .reset_index(drop=True)
)

print(f"Total series con >= {MIN_SEMANAS} semanas: {len(candidatos):,}")
candidatos.head(10)

## 4. Selección manual de 3 series para diversificar canal/región/categoría

In [ ]:
SERIES_OBJETIVO = [
    {
        "alias": "Uva Seedless · Feria libre · Maule",
        "producto_id": "Uva|Superior Seedless|Segunda | $/kilo | Feria libre",
        "region": "Región del Maule",
    },
    {
        "alias": "Chuleta centro · Supermercado · Arica",
        "producto_id": "Chuleta (centro) | $/kilo | Supermercado",
        "region": "Región de Arica y Parinacota",
    },
    {
        "alias": "Arándano · Feria libre · Metropolitana",
        "producto_id": "Arándano (blue)|Sin especificar|Segunda | $/kilo | Feria libre",
        "region": "Región Metropolitana de Santiago",
    },
]

# Validar que todas existen
for s in SERIES_OBJETIVO:
    serie = select_series(df, s["producto_id"], s["region"])
    print(f"{s['alias']}: {len(serie)} semanas, media CLP {serie.mean():,.0f}")

## 5. Pipeline de forecasting

Para cada serie:
1. Split temporal: 4 últimas semanas como test.
2. Ajustar Naive, ARIMA(1,1,1), Prophet sobre el train.
3. Evaluar con MAE / RMSE / MAPE.
4. Graficar pronósticos vs realidad.

In [ ]:
TEST_WEEKS = 4
resultados_globales = []
predicciones = {}

for s in SERIES_OBJETIVO:
    serie = select_series(df, s["producto_id"], s["region"])
    train, test = train_test_split_temporal(serie, test_weeks=TEST_WEEKS)
    
    modelos = {
        "Naive": forecast_naive(train, test.index),
        "ARIMA(1,1,1)": forecast_arima(train, test.index, order=(1, 1, 1)),
        "Prophet": forecast_prophet(train, test.index),
    }
    
    metricas_serie = []
    for nombre, res in modelos.items():
        m = evaluate(test, res.y_pred)
        m["modelo"] = nombre
        m["serie"] = s["alias"]
        metricas_serie.append(m)
    
    resultados_globales.extend(metricas_serie)
    predicciones[s["alias"]] = {"train": train, "test": test, "modelos": modelos}

metricas_df = pd.DataFrame(resultados_globales)[["serie", "modelo", "MAE", "RMSE", "MAPE_%"]]
metricas_df

## 6. Visualización de pronósticos

In [ ]:
fig, axes = plt.subplots(len(SERIES_OBJETIVO), 1, figsize=(13, 4 * len(SERIES_OBJETIVO)))
if len(SERIES_OBJETIVO) == 1:
    axes = [axes]

for ax, s in zip(axes, SERIES_OBJETIVO, strict=True):
    data = predicciones[s["alias"]]
    train, test, modelos = data["train"], data["test"], data["modelos"]
    
    ax.plot(train.index, train.values, label="Train (real)", color="steelblue", marker="o")
    ax.plot(test.index, test.values, label="Test (real)", color="black", marker="o", linewidth=2)
    
    colors = {"Naive": "gray", "ARIMA(1,1,1)": "tab:orange", "Prophet": "tab:green"}
    for nombre, res in modelos.items():
        ax.plot(res.y_pred.index, res.y_pred.values, label=nombre, color=colors[nombre], marker="x", linestyle="--")
        if res.y_lower is not None:
            ax.fill_between(res.y_pred.index, res.y_lower.values, res.y_upper.values,
                            color=colors[nombre], alpha=0.10)
    
    ax.set_title(s["alias"])
    ax.set_ylabel("Precio CLP")
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Resumen comparativo

Para cada serie, ¿qué modelo gana en MAPE?

In [ ]:
ranking = (
    metricas_df
    .sort_values(["serie", "MAPE_%"])
    .groupby("serie", as_index=False, sort=False)
    .first()
    [["serie", "modelo", "MAPE_%"]]
    .rename(columns={"modelo": "ganador"})
)
ranking

In [ ]:
# Promedio de cada modelo en las 3 series
promedio_modelo = (
    metricas_df.groupby("modelo")[["MAE", "RMSE", "MAPE_%"]]
    .mean()
    .sort_values("MAPE_%")
)
promedio_modelo

## 8. Conclusiones y limitaciones

_Completar después de ejecutar:_

**Lo que demuestra este notebook:**
- Flujo completo de forecasting reproducible: carga de datos limpios → selección de serie → split temporal → entrenamiento de múltiples modelos → evaluación cuantitativa → visualización con bandas de confianza.
- Comparación honesta contra un baseline ingenuo: si un modelo "sofisticado" no le gana al naive, no agrega valor.
- Diseño modular: la lógica vive en `src/model.py` y se reutiliza desde el notebook.

**Limitaciones a discutir abiertamente:**
1. **17 semanas de historia es muy poco.** Los intervalos de confianza de ARIMA y Prophet salen extremadamente amplios. Las métricas de test (4 puntos) son ruidosas.
2. **No se puede capturar estacionalidad anual** — no hay datos suficientes para detectar el patrón típico de subida de frutas/hortalizas según cosechas.
3. **El baseline naive es competitivo** en horizontes cortos sobre series cortas — esperable.
4. **Hiperparámetros fijos:** ARIMA(1,1,1) sin búsqueda automática; Prophet con configuración mínima. Con más datos, se justificaría `auto_arima` y tuning de Prophet.

**Próximos pasos posibles:**
- Extender el dataset con años anteriores (ej. precios 2020-2025) para habilitar Prophet con estacionalidad anual.
- Explorar regresión cross-product con LightGBM + SHAP para aprovechar las 100k filas como datos transversales.
- Detectar outliers de precio por producto-región como sistema de alerta más que como pronóstico.
- Empaquetar el pipeline en `dashboard/app.py` para que el usuario seleccione un producto y vea su pronóstico.